<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Name_Entity_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
dataset = load_dataset(
    "eriktks/conll2003",
    revision="convert/parquet"
)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
example=dataset['train'][0]
example

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [ ]:
dataset['train'].features['ner_tags']

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

In [ ]:
label_names=dataset['train'].features['ner_tags'].feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [ ]:
id2label={i:label for i, label in enumerate(label_names)}
id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC'}

In [ ]:
label2id={label:i for i, label in enumerate(label_names)}
label2id

{'O': 0,
 'B-PER': 1,
 'I-PER': 2,
 'B-ORG': 3,
 'I-ORG': 4,
 'B-LOC': 5,
 'I-LOC': 6,
 'B-MISC': 7,
 'I-MISC': 8}

In [ ]:
tokens=example['tokens']
ner_tags=example['ner_tags']
for token, ner_tag in zip(tokens, ner_tags):
  print(f"{token}\t{id2label[ner_tag]}")

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


In [ ]:
from transformers import AutoTokenizer
model_name='bert-base-cased'
tokenizer=AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
tokenized_inputs=tokenizer(
    tokens,
    is_split_into_words=True,
)

In [ ]:
tokenized_inputs[0]

Encoding(num_tokens=12, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [ ]:
bert_tokens=tokenizer.convert_ids_to_tokens(tokenized_inputs['input_ids'])
bert_tokens

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [ ]:
tokenized_inputs.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [ ]:
word_ids=tokenized_inputs.word_ids()
for token,word_id in zip(bert_tokens,word_ids):
  print(token,word_id)

[CLS] None
EU 0
rejects 1
German 2
call 3
to 4
boycott 5
British 6
la 7
##mb 7
. 8
[SEP] None


In [ ]:
def tokenize_and_aling_labels(examples):
  tokenized_inputs=tokenizer(
      examples['tokens'],
      truncation=True,
      is_split_into_words=True,
  )
  all_labels=examples['ner_tags']
  align_labels=[]
  for i, labels in enumerate(all_labels):
    word_ids=tokenized_inputs.word_ids(i)
    previous_word_idx=None
    label_id=[]
    for word_id in word_ids:
      if word_id is None or previous_word_id==word_id:
        label_id.append(-100)
      else:
        label_id.append(labels[word_id])
      previous_word_id=word_id
    align_labels.append(label_id)
  tokenized_inputs['labels']=align_labels
  return tokenized_inputs

In [ ]:
tokenized_dataset=dataset.map(
    tokenize_and_aling_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForTokenClassification

model=AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
import numpy as np
import evaluate

seqeval=evaluate.load('seqeval')

In [ ]:
def compute_metrics(eval_preds):
  logits,labels=eval_preds

  predictions=np.argmax(logits,axis=-1)

  true_predictions=[]
  true_labels=[]

  for pred, label in zip(predictions,labels):
    sentence_predictions=[]
    sentence_labels=[]
    for pred_id,label_id in zip(pred,label):
      if label_id!=-100:
        sentence_predictions.append(label_names[pred_id])
        sentence_labels.append(label_names[label_id])
    true_predictions.append(sentence_predictions)
    true_labels.append(sentence_labels)

  result=seqeval.compute(
      predictions=true_predictions,
      references=true_labels,
  )
  return {
      'precision':result['overall_precision'],
      'recall':result['overall_recall'],
      'f1':result['overall_f1'],
      'accuracy':result['overall_accuracy'],
  }

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 


KeyboardInterrupt: 

In [ ]:
import wandb

!wandb login
#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

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nihatqasimov (nihatqasimov-unec-business-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:

from transformers import TrainingArguments

training_args=TrainingArguments(
    output_dir='ner-bert',
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,

    eval_strategy='steps',
    save_strategy='steps',
    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model='f1',

    report_to='wandb',
    run_name='name-entity-recognition'
)

In [ ]:
from transformers import Trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nihatqasimov (nihatqasimov-unec-business-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
50,1.344284,0.246219,0.590221,0.623696,0.606497,0.932109
100,0.328474,0.113106,0.807792,0.837428,0.822343,0.970679
150,0.218137,0.084938,0.866190,0.876977,0.871550,0.978934
200,0.151329,0.066792,0.877913,0.900370,0.889000,0.982555
250,0.155746,0.058035,0.901596,0.912824,0.907175,0.984697
300,0.121177,0.053399,0.906861,0.920902,0.913828,0.985885
350,0.114974,0.049352,0.912546,0.925446,0.918951,0.986741
400,0.106575,0.049057,0.922332,0.929317,0.925811,0.987345
450,0.093268,0.046531,0.920725,0.932346,0.926499,0.987870
500,0.065334,0.043742,0.926615,0.939246,0.932888,0.988552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1317, training_loss=0.13414836002283234, metrics={'train_runtime': 781.6694, 'train_samples_per_second': 53.889, 'train_steps_per_second': 1.685, 'total_flos': 1050534559887048.0, 'train_loss': 0.13414836002283234, 'epoch': 3.0})

In [ ]:
import torch

def predict_ner_words(text, model, tokenizer, id2label):
    model.eval()
    device = next(model.parameters()).device

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        return_offsets_mapping=True
    )

    offset_mapping = encoding.pop("offset_mapping")[0]
    word_ids = encoding.word_ids(batch_index=0)

    model_inputs = {key: value.to(device) for key, value in encoding.items()}

    with torch.no_grad():
        outputs = model(**model_inputs)

    pred_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    word_predictions = []
    seen_word_ids = set()

    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        if word_id in seen_word_ids:
            continue

        seen_word_ids.add(word_id)


        subword_indices = [
            j for j, wid in enumerate(word_ids)
            if wid == word_id
        ]

        start = int(offset_mapping[subword_indices[0]][0])
        end = int(offset_mapping[subword_indices[-1]][1])

        word = text[start:end]

        label = id2label[int(pred_ids[subword_indices[0]])]

        word_predictions.append((word, label))

    return word_predictions

In [ ]:
text='The Amazon River flows through South America. Amazon Announced a new AI service yesterday.'

word_preds=predict_ner_words(
    text=text,
    model=trainer.model,
    tokenizer=tokenizer,
    id2label=id2label,
)

for word,label in word_preds:
  print(word,'--->',label)

The ---> O
Amazon ---> B-LOC
River ---> I-LOC
flows ---> O
through ---> O
South ---> B-LOC
America ---> I-LOC
. ---> O
Amazon ---> B-ORG
Announced ---> O
a ---> O
new ---> O
AI ---> B-MISC
service ---> O
yesterday ---> O
. ---> O
